# BÀI TẬP: TITANIC
**Nguồn:** kaggle.com/c/titanic (891 dòng)


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [ ]:
# A.1. Data size, column names, data types
print('Kích thước dữ liệu:', df.shape)

print('Tên cột:', list(df.columns))df.info()
print('\nThông tin kiểu dữ liệu:')

## A.2. Missing values & Duplicate data

In [ ]:
# A.2. Missing values & Duplicate data
print('Giá trị thiếu:\n', df.isnull().sum())

print('\nSố dòng trùng lặp:', df.duplicated().sum())print('Số dòng sau khi bỏ trùng:', df.drop_duplicates().shape[0])

## A.3. Invalid values

In [ ]:
# A.3. Invalid values
print('Số Age thiếu:', df['age'].isnull().sum())

print('Số embarked thiếu:', df['embarked'].isnull().sum())print(df[['age', 'fare']].describe())

print('Số fare âm:', (df['fare'] < 0).sum())print('\nMô tả nhanh của age và fare:\n')

## A.4. Create a new column
Tạo cột `family_size` = sibsp + parch + 1.

In [ ]:
# A.4. Create a new column
# family_size = sibsp + parch + 1

df['family_size'] = df['sibsp'] + df['parch'] + 1print(df[['sibsp', 'parch', 'family_size']].head())

---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [ ]:
# Group 1 — Central Tendency
num_cols = ['age', 'fare', 'family_size']

summary = df[num_cols].agg(['mean', 'median']).T    print(col, ':', df[col].mode().iloc[0])

summary.columns = ['mean', 'median']for col in num_cols:

print(summary)print('\nMode:')

## Group 2 — Dispersion

In [ ]:
# Group 2 — Dispersion
num_cols = ['age', 'fare', 'family_size']

print(df[num_cols].agg(['min', 'max', 'std', 'var']).T)    print(col, ':', round(q3 - q1, 2))

print('\nIQR:')    q3 = df[col].quantile(0.75)

for col in num_cols:    q1 = df[col].quantile(0.25)

## Group 3 — Location and Shape

In [ ]:
# Group 3 — Location and Shape
for col in ['age', 'fare', 'family_size']:

    print('\n===', col, '===')    print('Kurtosis:', round(df[col].kurt(), 4))

    print('Q1:', df[col].quantile(0.25))    print('Skewness:', round(df[col].skew(), 4))

    print('Median:', df[col].median())    print('Q3:', df[col].quantile(0.75))

---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Hạng vé nào có tỷ lệ sống sót cao nhất, chênh lệch bao nhiêu so với hạng thấp nhất?

In [ ]:
# Câu hỏi 1: Hạng vé nào có tỷ lệ sống sót cao nhất, chênh lệch bao nhiêu so với hạng thấp nhất?
class_survival = df.groupby('pclass')['survived'].mean().sort_values(ascending=False)

print(class_survival)print('Chênh lệch với hạng thấp nhất:', round((class_survival.max() - class_survival.min()) * 100, 2), '%')

print('\nHạng vé có tỷ lệ sống sót cao nhất:', class_survival.idxmax())print('Tỷ lệ sống sót cao nhất:', round(class_survival.max() * 100, 2), '%')

## Câu hỏi 2: Giới tính hay hạng vé ảnh hưởng đến sống sót mạnh hơn?

In [ ]:
# Câu hỏi 2: Giới tính hay hạng vé ảnh hưởng đến sống sót mạnh hơn?
sex_survival = df.groupby('sex')['survived'].mean()

class_survival = df.groupby('pclass')['survived'].mean()print('Hạng vé có tỷ lệ sống sót cao nhất:', class_survival.idxmax())

print('Tỷ lệ sống sót theo giới tính:')print('\nGiới tính có tỷ lệ sống sót cao nhất:', sex_survival.idxmax())

print(sex_survival)print(class_survival)
print('\nTỷ lệ sống sót theo hạng vé:')

## Câu hỏi 3: Vé đắt hơn có thực sự sống sót cao hơn không?

In [ ]:
# Câu hỏi 3: Vé đắt hơn có thực sự sống sót cao hơn không?
fare_survival = df.groupby(pd.cut(df['fare'], bins=[0, 20, 50, 100, 1000], right=False))['survived'].mean()

print(fare_survival)print('Tỷ lệ sống sót cao nhất ở nhóm vé:', fare_survival.idxmax())
print('\nNhóm vé đắt hơn có tỷ lệ sống sót cao hơn không?')

## Câu hỏi 4: Gia đình đông người có ảnh hưởng đến khả năng sống sót không?

In [ ]:
# Câu hỏi 4: Gia đình đông người có ảnh hưởng đến khả năng sống sót không?
family_survival = df.groupby('family_size')['survived'].mean()

print(family_survival)print('\nGia đình đông có tỷ lệ sống sót cao nhất ở family_size:', family_survival.idxmax())

## Câu hỏi 5: Cảng lên tàu (embark_town) nào có tỷ lệ sống sót cao nhất?

In [ ]:
# Câu hỏi 5: Cảng lên tàu (embark_town) nào có tỷ lệ sống sót cao nhất?
embark_survival = df.groupby('embark_town')['survived'].mean().sort_values(ascending=False)

print(embark_survival)print('\nCảng lên tàu có tỷ lệ sống sót cao nhất:', embark_survival.idxmax())

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể về dữ liệu Titanic.

*(Viết insight của bạn vào đây...)*

Tỷ lệ sống sót trên tàu Titanic khác nhau rõ rệt theo hạng vé và giới tính, trong đó phụ nữ và hành khách ở hạng vé cao có khả năng sống sót tốt hơn. Vé đắt hơn cũng cho thấy xu hướng sống sót cao hơn, nhưng yếu tố hạng vé và giới tính vẫn có tác động mạnh hơn. Số lượng thành viên trong gia đình có ảnh hưởng đến khả năng sống sót, nhưng không mạnh bằng yếu tố giới tính và hạng vé. Cảng lên tàu cũng tạo ra sự khác biệt nhất định trong tỷ lệ sống sót, cho thấy đặc điểm hành khách và cách lên tàu có thể ảnh hưởng tới kết quả. Nhìn chung, dữ liệu cho thấy những người thuộc nhóm có lợi thế xã hội và giới tính nữ có khả năng sống sót cao hơn.